# FreshMart Lab 1: Exploratory Data Analysis
**Azure Machine Learning (ทางเลือกฉุกเฉินแทน Fabric)**

การสำรวจข้อมูลเบื้องต้น (Exploratory Data Analysis) — ดูสถิติ กราฟ และค่าว่าง **ก่อน** สร้างโมเดล

บทบาท: **นักวิเคราะห์ข้อมูล** — สำรวจข้อมูลก่อนสร้างโมเดล  
จด **3 ข้อสังเกตสั้น ๆ** ส่ง Lab 2 (ไม่ต้องจำสูตรสถิติ)

### สิ่งที่แล็บนี้ตอบ
1. ข้อมูลขาดตรงไหน?
2. ของเสียสูงที่ประเภทสาขาไหน?
3. คนที่ Churn (เลิกซื้อ) พฤติกรรมต่างจากคนอยู่ต่ออย่างไร?

### ถ้าติด — อ่านก่อนถาม TA
| อาการ | ความหมาย |
| --- | --- |
| กราฟไม่ขึ้น | รอเซลล์ก่อนหน้าจบ แล้วรันใหม่ |
| ค่าสหสัมพันธ์ไม่ตรงทศนิยมทุกตัว | ปัด 2 ตำแหน่งใกล้เคียงพอ |
| ไม่เจอชั้น Bronze | กลับไป Lab 0 |


### เตรียมตัวโหลดข้อมูล

รันเซลล์ถัดไปเพื่อนิยาม `load_table_or_csv` — **อย่าข้าม**


In [ ]:
from pathlib import Path
import pandas as pd

BRONZE_TRANSACTIONS = "bronze/transactions"
BRONZE_CUSTOMERS = "bronze/customers"
SILVER_CUSTOMER_FEATURES = "silver/customer_features"
GOLD_PREDICTIONS = "gold/freshmart_predictions"
ASSET_BRONZE_TRANSACTIONS = "bronze-transactions"
ASSET_BRONZE_CUSTOMERS = "bronze-customers"
ASSET_SCORING_BATCH = "scoring-batch"
ASSET_SILVER_FEATURES = "silver-customer-features"
ASSET_GOLD_PREDICTIONS = "gold-freshmart-predictions"
TABLE_TO_ASSET = {
    BRONZE_TRANSACTIONS: ASSET_BRONZE_TRANSACTIONS,
    BRONZE_CUSTOMERS: ASSET_BRONZE_CUSTOMERS,
    SILVER_CUSTOMER_FEATURES: ASSET_SILVER_FEATURES,
    GOLD_PREDICTIONS: ASSET_GOLD_PREDICTIONS,
}
CSV_TO_ASSET = {
    "freshmart_transactions.csv": ASSET_BRONZE_TRANSACTIONS,
    "freshmart_customers.csv": ASSET_BRONZE_CUSTOMERS,
    "freshmart_scoring_batch.csv": ASSET_SCORING_BATCH,
}
RAW_FILES = (
    "freshmart_transactions.csv",
    "freshmart_customers.csv",
    "freshmart_scoring_batch.csv",
)


def _first_existing(paths):
    for path in paths:
        candidate = Path(path)
        if candidate.exists() and candidate.is_file():
            return candidate
    return None


def _writable_dir(path: Path) -> Path | None:
    try:
        path.mkdir(parents=True, exist_ok=True)
        probe = path / ".write_test"
        probe.write_text("ok", encoding="utf-8")
        probe.unlink()
        return path.resolve()
    except OSError:
        return None


def resolve_raw_csv(file_name: str) -> Path:
    found = _first_existing([
        Path("data") / "raw" / file_name,
        Path("data") / file_name,
        Path("../data") / "raw" / file_name,
        Path("../data") / file_name,
        Path("../../labs/data") / file_name,
        Path("../labs/data") / file_name,
        Path("labs/data") / file_name,
        Path(file_name),
    ])
    if found is None:
        raise FileNotFoundError(
            f"Cannot find {file_name}. Upload it to data/raw/ next to the notebook "
            "or clone this repo so labs/data/ is available."
        )
    return found


def load_csv(file_name: str) -> pd.DataFrame:
    asset_name = CSV_TO_ASSET.get(file_name)
    if asset_name:
        frame = load_data_asset(asset_name)
        if frame is not None:
            return frame
    found = resolve_raw_csv(file_name)
    print(f"Loaded CSV: {found}")
    return pd.read_csv(found)


def load_data_asset(asset_name: str):
    """Load a registered Azure ML data asset, or return None."""
    try:
        from azure.ai.ml import MLClient
        from azure.identity import DefaultAzureCredential

        ml_client = MLClient.from_config(credential=DefaultAzureCredential())
        asset = ml_client.data.get(name=asset_name, label="latest")
        path = asset.path
        print(f"Loaded data asset {asset_name} v{asset.version}: {path}")
        if str(path).lower().endswith(".parquet"):
            return pd.read_parquet(path)
        return pd.read_csv(path)
    except Exception as exc:
        print(f"Data asset '{asset_name}' unavailable ({exc})")
        return None


def register_data_asset(name: str, path, description: str = "") -> bool:
    """Register a file as an Azure ML data asset. Returns True when saved."""
    try:
        from azure.ai.ml import MLClient
        from azure.ai.ml.entities import Data
        from azure.ai.ml.constants import AssetTypes
        from azure.identity import DefaultAzureCredential

        ml_client = MLClient.from_config(credential=DefaultAzureCredential())
        asset = Data(
            name=name,
            path=str(path),
            type=AssetTypes.URI_FILE,
            description=description or f"FreshMart {name}",
        )
        created = ml_client.data.create_or_update(asset)
        print(f"Registered data asset {created.name} v{created.version}")
        return True
    except Exception as exc:
        print(f"Could not register data asset '{name}' ({exc})")
        return False


def resolve_artifact_root() -> Path:
    for candidate in (
        Path("data"),
        Path("../data"),
        Path("labs-azureml/data"),
    ):
        ready = _writable_dir(candidate)
        if ready is not None:
            return ready
    fallback = Path("data")
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback.resolve()


ARTIFACT_ROOT = resolve_artifact_root()


def layer_path(layer: str, stem: str, suffix: str = ".parquet") -> Path:
    folder = ARTIFACT_ROOT / layer
    folder.mkdir(parents=True, exist_ok=True)
    return folder / f"{stem}{suffix}"


def load_table_or_csv(table_name: str, file_name: str) -> pd.DataFrame:
    asset_name = TABLE_TO_ASSET.get(table_name)
    if asset_name:
        frame = load_data_asset(asset_name)
        if frame is not None:
            return frame
    layer, _, stem = table_name.partition("/")
    parquet = layer_path(layer, stem)
    if parquet.exists():
        frame = pd.read_parquet(parquet)
        print(f"Loaded parquet {parquet}: {len(frame):,} rows")
        return frame
    csv_fallback = ARTIFACT_ROOT / layer / f"{stem}.csv"
    if csv_fallback.exists():
        frame = pd.read_csv(csv_fallback)
        print(f"Loaded CSV artifact {csv_fallback}: {len(frame):,} rows")
        return frame
    print(f"Layer file '{table_name}' not found. Falling back to published CSV.")
    return load_csv(file_name)


def save_layer(frame: pd.DataFrame, table_name: str) -> Path:
    layer, _, stem = table_name.partition("/")
    parquet = layer_path(layer, stem)
    try:
        frame.to_parquet(parquet, index=False)
        print(f"Wrote {parquet} ({len(frame):,} rows)")
        return parquet
    except Exception as exc:
        csv_path = layer_path(layer, stem, suffix=".csv")
        frame.to_csv(csv_path, index=False)
        print(f"Parquet unavailable ({exc}). Wrote {csv_path}")
        return csv_path


### ขั้นตอนที่ 1: โหลดธุรกรรม

**โค้ดนี้ทำอะไร:** อ่าน `bronze/transactions` แล้วใช้ Pandas

- `shape` แสดง `(จำนวนแถว, จำนวนคอลัมน์)` — เมื่อถูกต้องควรได้ประมาณ `(3000, 14)`
- `head()` แสดง 5 แถวแรก เพื่อรู้จักคอลัมน์ เช่น `WasteUnits` (ของเสีย) และ `StoreType`


In [ ]:
df = load_table_or_csv(BRONZE_TRANSACTIONS, "freshmart_transactions.csv")
print(f"Pandas DataFrame shape: {df.shape}")
df.head()


### ขั้นตอนที่ 2: คุณภาพข้อมูล (ค่าว่าง)

**ความรู้จำเป็น**
- ค่าว่าง (missing) ทำให้โมเดล/สถิติเพี้ยน — ต้องรู้ก่อนแก้ใน Lab 2
- `isnull().sum()` นับแถวว่างต่อคอลัมน์

**ต้องเห็น:** `DiscountRate` ว่างประมาณ **89 แถว (~3%)**  
คอลัมน์อื่นไม่ควรว่างจำนวนมาก — ถ้าเป็นแบบนั้น แจ้ง TA


In [ ]:
print("=== DataFrame Info ===")
df.info()
print("\n=== Missing Values Count ===")
missing = df.isnull().sum()
print(missing[missing > 0])
print(f"DiscountRate missing rate: {df['DiscountRate'].isna().mean():.2%}")


### ขั้นตอนที่ 3: สถิติเชิงพรรณนา

**โค้ดนี้ทำอะไร:** `describe()` สรุป mean / min / max / เปอร์เซ็นไทล์

ดูคอลัมน์ `WasteUnits`: ค่าส่วนใหญ่ต่ำ แต่มีหางยาว (= เบ้ขวา) — ของเสียกระจุกบางวัน/บางสาขา


In [ ]:
df[["UnitsSold", "UnitPrice", "DiscountRate", "SalesAmount", "WasteUnits", "WasteCost"]].describe()


### ขั้นตอนที่ 4: Histogram — เห็นการกระจาย

**ความรู้จำเป็น**
- Histogram = นับความถี่ตามช่วงค่า
- เส้นโค้งบนกราฟช่วยดูรูปแบบการกระจาย

**ต้องเห็น:** `WasteUnits` เบ้ขวา (ค่าสูงมีน้อย)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["UnitsSold"], bins=15, kde=True, color="#00D2B4", ax=axes[0])
axes[0].set_title("Distribution of Units Sold")
sns.histplot(df["WasteUnits"], bins=10, kde=True, color="#F43F5E", ax=axes[1])
axes[1].set_title("Distribution of Waste Units (right-skewed)")
plt.tight_layout()
plt.show()


### Box plot ตามประเภทสาขา

**ความรู้จำเป็น:** Box plot เปรียบการกระจายระหว่างกลุ่มได้เร็ว

**อินไซต์ที่คาดหวัง:** Express ของเสียสูงกว่า Hypermarket โดยประมาณ  
(พื้นที่จัดเก็บจำกัด / ของเสียช่วงสุดสัปดาห์)

Hypermarket กับ Supermarket กล่องมักแบนใกล้ 0 เพราะวันส่วนใหญ่ไม่มีของเสีย — นั่นคือลักษณะข้อมูล ไม่ใช่กราฟพัง

ไม่ต้องได้ตัวเลขเป๊ะทุกทศนิยม — เห็นแนวโน้มถูกทางพอ


In [ ]:
plt.figure(figsize=(10, 5))
ax = sns.boxplot(data=df, x="StoreType", y="WasteUnits", hue="StoreType", palette="Set2", dodge=False)
legend = ax.get_legend()
if legend is not None:
    legend.remove()
plt.title("Waste Units by Store Type")
plt.show()


### ขั้นตอนที่ 5: Correlation

**ความรู้จำเป็น**
- ค่าใกล้ **+1** = ไปด้วยกัน, ใกล้ **-1** = สวนทาง, ใกล้ **0** = เกือบไม่เกี่ยว
- Heatmap คือตารางสหสัมพันธ์แบบสี

**ค่าอ้างอิงชุดนี้ (ปัด 2 ตำแหน่ง)**
- DiscountRate กับ UnitsSold ประมาณ **+0.38** (ลดราคาแล้วขายดีขึ้น)
- DiscountRate กับ WasteUnits ประมาณ **-0.12** (ลดราคามักเหลือทิ้งน้อยลง)


In [ ]:
numeric_cols = ["UnitsSold", "UnitPrice", "DiscountRate", "SalesAmount", "WasteUnits", "WasteCost", "IsWeekend"]
corr_matrix = df[numeric_cols].corr(numeric_only=True)
print(corr_matrix.round(2))

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1, linewidths=0.5)
plt.title("FreshMart Feature Correlation Matrix")
plt.show()


### ขั้นตอนที่ 6: สำรวจ Churn ของสมาชิก

**ความรู้จำเป็น**
- `Churn = 1` หมายถึงมีแนวโน้มยกเลิกหรือเลิกซื้อ, `0` หมายถึงอยู่ต่อ
- อัตรา Churn ชุดนี้อยู่ที่ประมาณ **19.3%** (290 จาก 1,500)
- `Age` ว่าง **37** แถว — Lab 2 จะเติมด้วยค่ามัธยฐาน (median)

Scatter ด้านล่าง: คนขาดซื้อนาน (`RecencyDays` สูง) และร้องเรียนบ่อย มักกระจุกที่ Churn=1


In [ ]:
df_cust = load_table_or_csv(BRONZE_CUSTOMERS, "freshmart_customers.csv")
print(f"Total Customers: {len(df_cust):,}")
print(df_cust["Churn"].value_counts(normalize=True).rename("rate"))
print(f"Age missing: {df_cust['Age'].isna().sum()} ({df_cust['Age'].isna().mean():.2%})")

plt.figure(figsize=(10, 5))
sns.scatterplot(
    data=df_cust,
    x="RecencyDays",
    y="ComplaintCount",
    hue="Churn",
    palette={0: "#00D2B4", 1: "#F43F5E"},
    alpha=0.7,
)
plt.title("Customer Churn Pattern: Recency vs Complaint Count")
plt.show()


### จุดตรวจ Lab 1

ผ่านแล้วพิมพ์ `Lab 1 verification passed`  
ก่อนไป Lab 2 จด 3 ข้อ: **เติม Age · แปลงหมวดหมู่ · ปรับสเกลเงินและความถี่**


In [ ]:
if len(df) != 3000:
    raise AssertionError(f"คาดว่าธุรกรรม 3,000 แถว แต่ได้ {len(df):,}")
if len(df_cust) != 1500:
    raise AssertionError(f"คาดว่าสมาชิก 1,500 แถว แต่ได้ {len(df_cust):,}")
if df["DiscountRate"].isna().sum() == 0:
    raise AssertionError("ชุดนี้ควรมี DiscountRate ว่าง เพื่อฝึกจัดการค่าว่างใน Lab 2")
if df_cust["Age"].isna().sum() == 0:
    raise AssertionError("ชุดนี้ควรมี Age ว่าง เพื่อฝึกเติมค่าใน Lab 2")
print("Lab 1 verification passed")
